# Lab 1: Build the HR policy index

## Business problem

Employees need answers from several approved HR policies. Before an LLM can answer, the documents must become searchable records that still identify their source and policy section.

## Mission

Turn the HR policy folder into a searchable index and verify that the correct evidence is retrieved for an employee question.

## Exercise 1: Open and read the supplied file

**Mission:** See what is actually inside the policy before writing any parsing code.

The file is in the same folder as this notebook. First read it as ordinary text.

In [ ]:
with open("hr_policy.txt", "r", encoding="utf-8") as file:
    policy_text = file.read()

print(policy_text)

### Inspection checkpoint

The full policy should appear. Confirm that the title, headings, and paragraphs were preserved before moving on.

## Exercise 2: Find the headings

**Mission:** Print only the lines that begin with `#`.

This lets us understand the document structure before we split it.

In [ ]:
for line in policy_text.splitlines():
    if line.startswith("#"):
        print(line)

## Exercise 3: Split the policy at its section headings

**Mission:** Turn each `##` heading and the text that follows it into one section record.

First we found the headings. Now we use the same marker to divide the document. We will print the section names so we can confirm the split before chunking.

In [ ]:
parts = policy_text.split("\n## ")

sections = []
for part in parts[1:]:
    heading, text = part.split("\n", 1)
    sections.append({"source": "hr_policy.txt", "section": heading, "text": text.strip()})

print("Sections found:", len(sections))
for section in sections:
    print(section["section"])

## Exercise 4: Chunk the sections without losing boundaries

**Mission:** Apply a chunking rule to each section while keeping short sections intact.

The splitter checks every section. If the section is shorter than the limit, it stays as one chunk. If it is longer than the limit, it is divided into smaller chunks with limited overlap.

This lab uses characters so the behavior is easy to inspect. Production systems may use tokens, but the same idea applies.

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

CHUNK_SIZE = 500
CHUNK_OVERLAP = 50

splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    separators=["\n\n", "\n", ". ", " ", ""],
)

### Apply the chunker

**Mission:** Run the configured splitter against the sections and inspect how many chunks each section produces.

The previous cell only defined the rules. This next cell applies them to the data.

In [ ]:
chunks = []

for section in sections:
    pieces = splitter.split_text(section["text"])

    for position, piece in enumerate(pieces):
        chunks.append({**section, "text": piece, "position": position})

for chunk in chunks:
    print(chunk["section"], "| position", chunk["position"], "|", len(chunk["text"]), "characters")

### Chunking checkpoint

Short sections should remain intact. If a section is split, every resulting chunk should retain the same source and section. Overlap is used only when a split occurs.

## Exercise 5: Inspect one embedding

**Mission:** Convert one chunk into the numeric representation used for similarity search.

**Why it matters:** The index and employee questions must use the same embedding model and compatible dimensions.

**Industry choices:** This lab uses OpenAI `text-embedding-3-small` because it is recognizable in job descriptions and simple to call. Other commonly encountered choices include Cohere Embed, Voyage AI embeddings, Google Gemini embeddings, and open-weight models such as BGE or E5. Choose one provider for an index and do not mix vector dimensions.

In [ ]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
client = OpenAI()
EMBEDDING_MODEL = "text-embedding-3-small"

response = client.embeddings.create(model=EMBEDDING_MODEL, input=chunks[0]["text"])
vector = response.data[0].embedding

print("Vector dimensions:", len(vector))
print("First 10 values:", vector[:10])

## Exercise 6: Build the vector index

**Mission:** Store every chunk together with the metadata needed for filtering and citations.

**Industry choices:** This lab uses LangChain's `InMemoryVectorStore` so the retrieval steps remain visible. Production teams commonly use PostgreSQL with pgvector, Pinecone, Qdrant, Weaviate, Milvus, or a cloud search service such as Azure AI Search. The choice depends on persistence, scale, hybrid search, access control, and operating model.

In [ ]:
from langchain_core.documents import Document
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_openai import OpenAIEmbeddings

documents = []
for chunk in chunks:
    metadata = {
        "source": chunk["source"],
        "section": chunk["section"],
        "position": chunk["position"],
        "version": "2026.1",
        "status": "current",
    }
    documents.append(Document(page_content=chunk["text"], metadata=metadata))

embeddings = OpenAIEmbeddings(model=EMBEDDING_MODEL)
vector_store = InMemoryVectorStore.from_documents(documents, embedding=embeddings)
print("Indexed chunks:", len(documents))

## Exercise 7: Retrieve and inspect evidence

**Mission:** Verify which policy evidence would be sent to an LLM.

Change `TOP_K` and inspect the effect. More retrieved chunks can add useful evidence or unrelated material.

In [ ]:
QUESTION = "Can I expense a $300 train ticket without approval?"
TOP_K = 3

results = vector_store.similarity_search_with_score(QUESTION, k=TOP_K)

for document, score in results:
    print("Similarity:", round(score, 3))
    print("Source:", document.metadata["source"])
    print("Section:", document.metadata["section"])
    print(document.page_content)
    print()

## Exercise 8: Record the index configuration

**Mission:** Record enough configuration to rebuild or troubleshoot this index later.

An index depends on the source files, parser, chunk settings, embedding model, and index version.

In [ ]:
INDEX_CONFIG = {
    "index_version": "2026.1",
    "parser": "Markdown headings and paragraphs",
    "chunk_size_characters": CHUNK_SIZE,
    "chunk_overlap_characters": CHUNK_OVERLAP,
    "embedding_model": EMBEDDING_MODEL,
}

for name, value in INDEX_CONFIG.items():
    print(name, "=", value)

### Operations checkpoint

If any of these settings change, rebuild the index and run the retrieval tests again. This is the same idea as recording the software version and configuration before a network change.

## Exercise 9: Inspect table-shaped evidence

**Mission:** Keep a policy amount connected to its unit and condition.

A parser that returns `60` without `meals`, `per day`, or `receipt required` has lost meaning.

In [ ]:
expense_rows = [
    {"expense": "Domestic meals", "limit": "USD 60 / day", "condition": "Receipt required"},
    {"expense": "Rail travel", "limit": "USD 250 / trip", "condition": "Approval above limit"},
]

for row in expense_rows:
    print(row["expense"], "|", row["limit"], "|", row["condition"])

## Exercise 10: Test an exact identifier

**Mission:** See why a vector search alone may miss an error code or plan code.

In [ ]:
identifier_records = [
    "E-4042: invalid customer ID",
    "Common troubleshooting steps for customer errors",
    "Contact support when an error continues",
]

question = "What does error code E-4042 mean?"
keyword_matches = [record for record in identifier_records if "E-4042" in record]
print("Keyword matches:", keyword_matches)

### Lab 1 checkpoint

`reimbursements.md` and `Travel approval` should rank first. You can now trace a retrieval result through its embedding configuration, chunk, section, source, and evidence shape.